In [ ]:
import os
import re
import json
import time
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# -------------------- Настройки --------------------
yandex = YandexTranslate()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [3]:
SOURCE_REPO_ID = "DeepPavlov/coral"
LOCAL_SAVE_PATH = "./coral_ru"
CACHE_FILE = "translation_cache_coral.jsonl"
CONFIGS_TO_TRANSLATE = ['corpus', 'queries', 'rewritten_queries']  # qrels не переводим
SPLITS = ['train', 'test']

In [ ]:
# -------------------- Кэш --------------------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()
print(f"Записей в кэше: {len(translation_cache)}")

# -------------------- Улучшенный переводчик (re + длинные тексты) --------------------
def is_numeric_string(s: str) -> bool:
    """True, если в строке нет ни одной буквы (числа, даты, диапазоны)."""
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    # ---------- базовые проверки ----------
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    # ---------- попытка перевода целиком ----------
    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if '413' in error_str or 'too long' in error_str:
                break   # дальше будем разбивать
            if any(c in error_str for c in ['502','503','504']):
                time.sleep(delay * (attempt+1))
            elif '429' in error_str:
                time.sleep(delay*4 + 10)
            else:
                time.sleep(delay)

    # ---------- разбиение и посегментный перевод ----------
    # 1) Разбиваем на предложения
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1:
        return _translate_by_parts(sentences, retries, delay, text)

    # 2) Текст длиннее 4000 символов – пробуем разбить по менее значимым разделителям
    if len(text) > 4000:
        sub_parts = re.split(r'(?<=[;:,])\s+', text)
        if len(sub_parts) > 1:
            return _translate_by_parts(sub_parts, retries, delay, text)
        # Если не помогло – режем принудительно
        return _translate_by_force_split(text, 3500, retries, delay)

    # 3) Короткий текст, который не удалось перевести – возвращаем оригинал
    return text, True


def _translate_by_parts(parts, retries, delay, original_full_text):
    """Переводит список фрагментов, склеивает и возвращает полный перевод."""
    translated_parts = []
    for part in parts:
        if not part.strip():
            translated_parts.append(part)
            continue
        t, ok = translate_text_robust(part, retries, delay)
        if not ok:
            t = part   # fallback к оригиналу
        translated_parts.append(t)
    full = ' '.join(translated_parts)
    translation_cache[original_full_text] = full
    append_cache(original_full_text, full)
    return full, True


def _translate_by_force_split(text, chunk_size=3500, retries=3, delay=3):
    """Принудительно режет текст на блоки ~chunk_size символов."""
    words = text.split()
    chunks = []
    current_chunk = []
    current_len = 0
    for word in words:
        current_chunk.append(word)
        current_len += len(word) + 1
        if current_len >= chunk_size:
            chunks.append(' '.join(current_chunk))
            current_chunk = []
            current_len = 0
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    # Если после разбиения получился один чанк (текст и так короткий) – не переводим
    if len(chunks) <= 1:
        return text, True

    logging.info(f"Force splitting into {len(chunks)} chunks")

    translated_chunks = []
    for chunk in chunks:
        t, ok = translate_text_robust(chunk, retries, delay)
        if not ok:
            t = chunk
        translated_chunks.append(t)
    full = ' '.join(translated_chunks)
    translation_cache[text] = full
    append_cache(text, full)
    return full, True

# -------------------- Функция для перевода списка сообщений --------------------
def translate_dialog_messages(dialog_list):
    """
    Переводит только поле 'content' внутри каждого сообщения.
    Возвращает список с переведёнными content и оригинальными остальными полями.
    """
    if not dialog_list:
        return [], True

    translated = []
    all_ok = True
    for msg in dialog_list:
        new_msg = dict(msg)   # копируем все поля (role, ...)
        if 'content' in msg and isinstance(msg['content'], str):
            cont, ok = translate_text_robust(msg['content'])
            if not ok:
                all_ok = False
            new_msg['content'] = cont if ok else msg['content']
        translated.append(new_msg)
    return translated, all_ok

# -------------------- Перевод одного примера --------------------
def translate_example(example, text_columns):
    """
    text_columns – список полей, которые нужно перевести.
    Если значение – строка, переводим как строку.
    Если значение – список сообщений, переводим 'content' в каждом сообщении.
    """
    result = dict(example)
    success = True

    for col in text_columns:
        value = example.get(col)
        if isinstance(value, str):
            trans, ok = translate_text_robust(value)
            if not ok:
                success = False
            result[col + '_ru'] = trans if ok else value
        elif isinstance(value, list) and value and isinstance(value[0], dict):
            # Список сообщений – переводим только content
            trans_list, ok = translate_dialog_messages(value)
            if not ok:
                success = False
            result[col + '_ru'] = trans_list
        else:
            # Для нестроковых полей просто копируем
            result[col + '_ru'] = value

    result['_success'] = success
    return result

# -------------------- Обработка сплита --------------------
def process_config_split(config_name, split_name, source_split, fields, progress_file):
    translated_records = []
    failed_indices = set()

    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get('_failed', False):
                        failed_indices.add(rec['_index'])
                    else:
                        translated_records.append(rec)
                except:
                    continue
        logging.info(f"[{config_name}/{split_name}] Resuming: {len(translated_records)} ok, {len(failed_indices)} failed")

    start_index = len(translated_records) + len(failed_indices)
    total = len(source_split)

    if start_index < total:
        logging.info(f"[{config_name}/{split_name}] Starting from index {start_index}...")
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_split.select(range(start_index, total))),
                desc=f"Translating {config_name}/{split_name}",
                total=total - start_index
            )
            for idx, example in pbar:
                global_idx = start_index + idx
                if global_idx in {r.get('_index', -1) for r in translated_records}:
                    continue

                translated = translate_example(example, fields)
                record_out = {
                    '_index': global_idx,
                    '_failed': not translated['_success'],
                    **translated
                }
                del record_out['_success']

                f.write(json.dumps(record_out, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(0.5)

                if not record_out['_failed']:
                    translated_records.append(record_out)
                else:
                    failed_indices.add(global_idx)

    # Сбор успешных записей
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if not rec.get('_failed', False):
                        rec.pop('_index', None)
                        rec.pop('_failed', None)
                        all_successful.append(rec)
                except:
                    continue

    if not all_successful:
        logging.error(f"[{config_name}/{split_name}] No successful records!")
        return None

    ok_cnt = len(all_successful)
    fail_cnt = total - ok_cnt
    logging.info(f"[{config_name}/{split_name}] Done: {ok_cnt}/{total} ({fail_cnt} failed)")
    if fail_cnt > 0:
        logging.info(f"Чтобы повторить failed, удалите или измените {progress_file} и запустите снова.")
    return Dataset.from_list(all_successful)

In [ ]:
# -------------------- Запуск --------------------
logging.info("Loading Coral dataset...")
all_configs = {}

for config in CONFIGS_TO_TRANSLATE:
    source = load_dataset(SOURCE_REPO_ID, config)
    config_splits = {}

    # Определяем поля для перевода
    if config == 'corpus':
        fields = ['text']
    elif config in ['queries', 'rewritten_queries']:
        fields = ['text']
    else:
        fields = []

    for split in SPLITS:
        if split not in source:
            logging.warning(f"Split '{split}' not found in {config}, skipping")
            continue
        progress_file = f"translated_coral_{config}_{split}.jsonl"
        ds = process_config_split(config, split, source[split], fields, progress_file)
        if ds is not None:
            config_splits[split] = ds
    if config_splits:
        all_configs[config] = DatasetDict(config_splits)

# Добавляем qrels без перевода
try:
    qrels = load_dataset(SOURCE_REPO_ID, 'qrels')
    all_configs['qrels'] = qrels
    print("qrels добавлены без перевода")
except Exception as e:
    logging.warning(f"Не удалось загрузить qrels: {e}")

# Сохраняем и сравниваем
if all_configs:
    final_dataset = DatasetDict(all_configs)
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nСохранено в {LOCAL_SAVE_PATH}")

    # Сравнение с оригиналом
    print("\n" + "="*60)
    print("СРАВНЕНИЕ С ОРИГИНАЛОМ")
    print("="*60)
    for config in all_configs:
        orig = load_dataset(SOURCE_REPO_ID, config)
        for split in orig:
            our_len = len(all_configs[config][split])
            orig_len = len(orig[split])
            status = "✅" if our_len == orig_len else "❌"
            print(f"{status} {config}/{split}: orig={orig_len}, ours={our_len}")
else:
    print("Не удалось перевести ни одной конфигурации")

In [ ]:
import os, re, json, time, logging, gc
from datasets import load_dataset, Dataset, DatasetDict
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# -------------------- Настройки --------------------
yandex = YandexTranslate()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

CACHE_FILE = "translation_cache_coral.jsonl"
PROGRESS_DIR = "."
CHUNK_SIZE = 3500

# Какие сплиты и конфигурации осталось перевести
# Какие сплиты и конфигурации осталось перевести
TO_FINISH = {
    'queries': ['test']                    # train уже готов, остался только test
}

# -------------------- Кэш --------------------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()
print(f"Записей в кэше: {len(translation_cache)}")

# -------------------- Улучшенный переводчик --------------------
def is_numeric_string(s: str) -> bool:
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    # Попытка перевода целиком
    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if '413' in error_str or 'too long' in error_str:
                break
            if any(c in error_str for c in ['502','503','504']):
                time.sleep(delay * (attempt+1))
            elif '429' in error_str:
                time.sleep(delay*4 + 10)
            else:
                time.sleep(delay)

    # Разбиение на предложения
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1:
        return _translate_parts(sentences, retries, delay, text)

    # Длинный текст — принудительное разбиение
    if len(text) > 4000:
        return _force_split(text, CHUNK_SIZE, retries, delay)

    # Короткий текст, который не перевёлся — возвращаем оригинал
    return text, True

def _translate_parts(parts, retries, delay, original):
    translated = []
    for part in parts:
        if not part.strip():
            translated.append(part)
            continue
        t, _ = translate_text_robust(part, retries, delay)
        translated.append(t if t else part)
    full = ' '.join(translated)
    translation_cache[original] = full
    append_cache(original, full)
    return full, True

def _force_split(text, chunk_size, retries, delay):
    words = text.split()
    chunks = []
    cur_chunk = []
    cur_len = 0
    for word in words:
        cur_chunk.append(word)
        cur_len += len(word) + 1
        if cur_len >= chunk_size:
            chunks.append(' '.join(cur_chunk))
            cur_chunk = []
            cur_len = 0
    if cur_chunk:
        chunks.append(' '.join(cur_chunk))

    if len(chunks) <= 1:
        return text, True

    logging.info(f"Force splitting into {len(chunks)} chunks")
    translated = []
    for chunk in chunks:
        t, _ = translate_text_robust(chunk, retries, delay)
        translated.append(t if t else chunk)
    full = ' '.join(translated)
    translation_cache[text] = full
    append_cache(text, full)
    return full, True

# -------------------- Перевод сообщений диалога --------------------
def translate_dialog(dialog_list):
    if not dialog_list:
        return [], True
    result = []
    for msg in dialog_list:
        new_msg = dict(msg)
        if 'content' in msg and isinstance(msg['content'], str):
            cont, _ = translate_text_robust(msg['content'])
            new_msg['content'] = cont if cont else msg['content']
        result.append(new_msg)
    return result, True

# -------------------- Низкопамятная обработка сплита --------------------
def count_lines(filename):
    if not os.path.exists(filename):
        return 0
    cnt = 0
    with open(filename, "r", encoding="utf-8") as f:
        for _ in f:
            cnt += 1
    return cnt

def process_split_low_memory(config_name, split_name, source_split, progress_file):
    total = len(source_split)
    start_idx = count_lines(progress_file)

    if start_idx >= total:
        logging.info(f"[{config_name}/{split_name}] Уже завершено.")
        return

    logging.info(f"[{config_name}/{split_name}] Продолжаем с {start_idx}, всего {total}")

    with open(progress_file, "a", encoding="utf-8") as out:
        pbar = tqdm(
            enumerate(source_split.select(range(start_idx, total))),
            desc=f"Translating {config_name}/{split_name}",
            total=total - start_idx
        )
        for i, example in pbar:
            # Переводим text (строка или список сообщений)
            value = example.get('text')
            if isinstance(value, str):
                trans, _ = translate_text_robust(value)
                text_ru = trans if trans else value
            elif isinstance(value, list):
                trans_list, _ = translate_dialog(value)
                text_ru = trans_list
            else:
                text_ru = value

            record = {
                '_index': start_idx + i,
                '_failed': False,
                'text': value,
                'text_ru': text_ru,
                **{k: v for k, v in example.items() if k != 'text'}
            }

            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

            # Каждые 500 записей освобождаем память
            if (i + 1) % 500 == 0:
                gc.collect()

    logging.info(f"[{config_name}/{split_name}] Готово.")

# -------------------- Запуск --------------------
# Загружаем только те конфигурации, которые будем переводить
source = {}
for config in TO_FINISH:
    source[config] = load_dataset("DeepPavlov/coral", config)

for config, splits in TO_FINISH.items():
    for split in splits:
        if split not in source[config]:
            logging.warning(f"Сплит {split} не найден в {config}")
            continue
        progress_file = f"translated_coral_{config}_{split}.jsonl"
        process_split_low_memory(config, split, source[config][split], progress_file)

print("\nВсе оставшиеся сплиты переведены!")
print("Теперь можно собрать финальный датасет и залить на HF.")

In [ ]:
import os

# Ожидаемое количество строк
EXPECTED = {
    'corpus':    {'train': 201196},
    'queries':   {'train': 59592, 'test': 6480},
    'qrels':     {'train': 59592, 'test': 6480}
}

print("=" * 60)
print("СРАВНЕНИЕ КОЛИЧЕСТВА СТРОК")
print("=" * 60)

all_ok = True
for config, splits in EXPECTED.items():
    for split, expected_count in splits.items():
        fname = f"translated_coral_{config}_{split}.jsonl"
        if not os.path.exists(fname):
            print(f"{config}/{split}: файл не найден")
            all_ok = False
            continue
        
        count = 0
        with open(fname, 'r', encoding='utf-8') as f:
            for _ in f:
                count += 1
        
        if count == expected_count:
            print(f"{config}/{split}: {count}")
        else:
            print(f"{config}/{split}: {count} (ожидалось {expected_count})")
            all_ok = False

print("=" * 60)
if all_ok:
    print("corpus и queries готовы!")
else:
    print("Есть расхождения в corpus/queries.")

In [ ]:
import json, os
from datasets import Dataset, DatasetDict, load_dataset

# -------------------- 1. Загружаем qrels из оригинала --------------------
print("Загружаем qrels из оригинального датасета...")
qrels = load_dataset("DeepPavlov/coral", "qrels")
print(f"  qrels/train: {len(qrels['train'])}")
print(f"  qrels/test: {len(qrels['test'])}")

# -------------------- 2. Собираем corpus и queries из прогресс-файлов --------------------
final = {}
for config in ['corpus', 'queries']:
    config_splits = {}
    for split in EXPECTED[config]:
        fname = f"translated_coral_{config}_{split}.jsonl"
        if not os.path.exists(fname):
            print(f" Файл {fname} не найден, пропускаем")
            continue
        
        with open(fname, 'r', encoding='utf-8') as f:
            records = [json.loads(line) for line in f]
        for r in records:
            r.pop('_index', None)
            r.pop('_failed', None)
            r.pop('_success', None)
        config_splits[split] = Dataset.from_list(records)
        print(f"  {config}/{split}: {len(records)} записей")
    
    final[config] = DatasetDict(config_splits)

# -------------------- 3. Добавляем qrels --------------------
final['qrels'] = qrels

# -------------------- 4. Сохраняем --------------------
final_dataset = DatasetDict(final)
final_dataset.save_to_disk("./coral_ru")
print("\nДатасет сохранён в ./coral_ru")

# -------------------- 5. Финальная проверка --------------------
print("\n" + "=" * 60)
print("ФИНАЛЬНАЯ ПРОВЕРКА")
print("=" * 60)
for config in ['corpus', 'queries', 'qrels']:
    orig = load_dataset("DeepPavlov/coral", config)
    for split in orig:
        our = len(final_dataset[config][split])
        orig_len = len(orig[split])
        status = "✅" if our == orig_len else "❌"
        print(f"{status} {config}/{split}: orig={orig_len}, ours={our}")

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

import os
from datasets import DatasetDict

def load_nested_dataset_dict(path):
    result = DatasetDict()
    for config_name in os.listdir(path):
        config_path = os.path.join(path, config_name)
        if not os.path.isdir(config_path):
            continue
        if os.path.exists(os.path.join(config_path, "dataset_dict.json")):
            result[config_name] = DatasetDict.load_from_disk(config_path)
    return result

final_dataset = load_nested_dataset_dict("./coral_ru")

REPO_ID = "DeepPavlov/coral_ru"

for config_name, ds_dict in final_dataset.items():
    ds_dict.push_to_hub(
        REPO_ID,
        config_name=config_name,
        private=False,
        commit_message="Russian translation – corpus & queries translated, qrels original"
    )
    print(f"{config_name} uploaded")

print(f"Готово: https://huggingface.co/datasets/{REPO_ID}")